# Conway's Game of Life (NumPy) — Guided Lab

In this lab you will use NumPy to implement Conway's Game of Life on a 2D grid.

> **Attribution:** This exercise is inspired by and adapted from Nicolas P. Rougier's *NumPy tutorial*.  We borrowed the core ideas (vectorized neighborhood counting & rules) and reshaped them into this assignment.


## Learning Objectives

By the end of this assignment, you should be able to:

- Create and manipulate NumPy arrays for 2D grids.
- Implement classic **Game of Life** rules using **vectorized** array operations.
- Cleanly separate concerns across helper functions:
  - `make_board` → initialize a board
  - `patterns` → provide named seed patterns (including a **random** generator)
  - `seed_board` → build a board and place/produce the chosen pattern
  - `center_place` → copy a smaller pattern into the center of a larger board
  - `iterate` → compute one simulation step with a **fixed zero border**
- Run a simple **text/HTML** animation right in the notebook.


## How to Use This Notebook

- Each **Problem** section includes:
  1. **Goal & Spec**: what the function must do
  2. **Steps**: a suggested plan
  3. **Starter Code**: edit where marked `# TODO`
  4. **Quick Checks**: small tests to sanity-check your work
- Keep the **function names & signatures exactly** as given.
- Do not modify the **Provided Utilities** cell (render/animation helpers).
- The border of the board is treated as **always dead** (fixed zeros). Your logic must **not** write outside the inner area.
- You can re-run the **Demo** at the end to visualize patterns.


In [ ]:
# == Imports (Provided) ==
import time
import numpy as np
from IPython.display import display, HTML, clear_output
from ipywidgets import HTML, Layout

## Background: Conway's Game of Life

**General premise.** The Game of Life is a famous zero-player cellular automaton devised by mathematician John Conway. A population of cells lives on a 2D grid; each cell is either **alive (1)** or **dead (0)**. Time advances in **discrete steps**. At each step, every cell updates *simultaneously* according to simple local rules that depend only on its eight neighbors.

**Setup.** We represent the world as a NumPy array where `0` means dead and `1` means alive. In this assignment, we use a **fixed zero border** (the outermost rows/columns are always 0), which keeps the focus on vectorized updates without worrying about wrap-around edges.

**Rules (applied to each *inner* cell every step):**
- **Birth:** A dead cell with exactly **3** live neighbors becomes alive.
- **Survival:** A live cell with **2 or 3** live neighbors stays alive.
- **Otherwise:** The cell becomes (or remains) dead.

**Why this matters.** Despite the rules being tiny, the resulting behavior is rich: still lifes, oscillators, spaceships, and complex emergent patterns. In computer science and the wider scientific community, Life is used to illustrate **emergence**, **parallel computation**, **vectorization**, **simulation**, and **scientific visualization**. In this lab, you'll practice these ideas while writing clean, testable NumPy code.


---

## Problem 1 — `make_board(size=32)`

### Goal & Spec
Create and return a **square** NumPy array of shape `(size, size)` filled with zeros of `dtype` `int`.  
This will represent the Game of Life grid (0 = dead, 1 = alive).

### Hint
1. `np.zeros` will be very useful here ...

### Starter Code


In [ ]:
def make_board(size: int = 32) -> np.ndarray:
    """Return a (size x size) zero-initialized integer grid."""
    return np.zeros((size, size), dtype=int)


### Quick Checks
Run these after you implement `make_board`.


In [ ]:
# Basic sanity checks for make_board
try:
    _b = make_board(5)
    assert isinstance(_b, np.ndarray)
    assert _b.shape == (5, 5)
    assert _b.dtype == int or _b.dtype == np.int64 or _b.dtype == np.int32
    assert np.all(_b == 0)
    print("✅ make_board basic checks passed!")
except AssertionError:
    print("❌ make_board failed a check. Review your implementation.")
    print("Check that implementation is an int type.")
except NotImplementedError as e:
    print(e)


✅ make_board basic checks passed!


---

## Problem 2 — `patterns()`

### Goal & Spec
Return a **dictionary** of named patterns used to seed the board.

- Keys are **strings** (pattern names).
- Values are either:
  - **NumPy arrays** of 0/1 (for fixed patterns), or
  - **Callables** that accept a shape (tuple) and return a **NumPy array** (for generated patterns like `random`).

For this assignment:
- We **provide** several fixed patterns already implemented: (e.g., `loaf` and `pulsar`).
- You must implement two additional patterns.
    - `beehive`

    <img src="https://upload.wikimedia.org/wikipedia/commons/6/67/Game_of_life_beehive.svg"
     alt="Game of Life Beehive"
     style="width:50%;">

    - `beacon`

     <img src="https://upload.wikimedia.org/wikipedia/commons/1/1c/Game_of_life_beacon.gif"
     alt="Game of Life Beacon"
     style="width:50%;">

    
- You must implement the function `random_pattern` which, given a `shape` (e.g., `(size, size)`), returns a 0/1 array of that shape with random values.  It is important that the first and last row, and the first and last columns are set to zero (these represent the dead border cells)

> Implementation note: returning a callable for `"random"` lets other functions (like `seed_board`) request the appropriate shape without hard-coding size here.

### Hint
1. np.random will be useful in `random_pattern`


In [ ]:
def patterns() -> dict:
    """
    Return a mapping of pattern name -> array (for fixed patterns) or callable(shape)->array (for generated patterns).
    Provided fixed patterns: 'loaf' and 'pulsar'.
    Required generated pattern: 'random'.
    """
    # --- Provided: fixed patterns  ---
    loaf = np.array([
        [0,1,1,0],
        [1,0,0,1],
        [0,1,0,1],
        [0,0,1,0]
    ], dtype=int)

    blinker = np.array([[1,1,1]], dtype=int)

    toad = np.array([[0,1,1,1],
                     [1,1,1,0]], dtype=int)

    block = np.array([[1,1],[1,1]], dtype=int)

    boat = np.array([[1,1,0],
                     [1,0,1],
                     [0,1,0]], dtype=int)

    tub = np.array([[0,1,0],
                    [1,0,1],
                    [0,1,0]], dtype=int)

    pulsar = np.array([
        [0,0,1,1,0,0,0,0,0,1,1,0,0],
        [0,0,0,1,1,0,0,0,1,1,0,0,0],
        [1,0,0,1,0,1,0,1,0,1,0,0,1],
        [1,1,1,0,1,1,0,1,1,0,1,1,1],
        [0,1,0,1,0,1,0,1,0,1,0,1,0],
        [0,0,1,1,1,0,0,0,1,1,1,0,0],
        [0,0,0,0,0,0,0,0,0,0,0,0,0],
        [0,0,1,1,1,0,0,0,1,1,1,0,0],
        [0,1,0,1,0,1,0,1,0,1,0,1,0],
        [1,1,1,0,1,1,0,1,1,0,1,1,1],
        [1,0,0,1,0,1,0,1,0,1,0,0,1],
        [0,0,0,1,1,0,0,0,1,1,0,0,0],
        [0,0,1,1,0,0,0,0,0,1,1,0,0]
    ], dtype=int)

    # --- Students: implement these fixed patterns ---
    beehive = np.array([
        [0,1,1,0],
        [1,0,0,1],
        [0,1,1,0]
    ], dtype=int)

    beacon = np.array([
        [1,1,0,0],
        [1,1,0,0],
        [0,0,1,1],
        [0,0,1,1]
    ], dtype=int)

    # --- Required: generated pattern (implement this) ---
    def random_pattern(shape):
        """
        Return a random 0/1 ndarray of the given shape.
        - shape: tuple like (rows, cols)
        """
        grid = np.random.randint(2, size=shape, dtype=int)
        # Set borders to 0
        grid[0, :] = 0
        grid[-1, :] = 0
        grid[:, 0] = 0
        grid[:, -1] = 0
        return grid

    # Build and return the mapping
    pats = {
        "blinker": blinker,
        "toad": toad,
        "beacon": beacon,
        "pulsar": pulsar,
        "block": block,
        "beehive": beehive,
        "loaf": loaf,
        "boat": boat,
        "tub": tub,
        "random": random_pattern,   # <- callable function
    }

    return pats


### Quick Checks
Run these after you implement `patterns` (and `random_pattern` inside it).


In [ ]:
try:
    _p = patterns()
    assert "beacon" in _p and "beehive" in _p and "random" in _p
    # check 'beacon' and 'beehive'
    assert isinstance(_p["beacon"], np.ndarray) and _p["beacon"].dtype == int
    assert isinstance(_p["beehive"], np.ndarray) and _p["beehive"].dtype == int

    # Check random callable and properties
    rnd = _p["random"]
    assert callable(rnd)
    _r_small = rnd((5,5))
    assert isinstance(_r_small, np.ndarray) and _r_small.shape == (5,5)
    assert set(np.unique(_r_small)).issubset({0,1})

    # Additional checks for border zeros and sufficient density on a larger sample
    _r = rnd((10,10))
    assert np.all(_r[0, :] == 0) and np.all(_r[-1, :] == 0)
    assert np.all(_r[:, 0] == 0) and np.all(_r[:, -1] == 0)
    total_cells = _r.size
    assert _r.sum() > 0.2 * total_cells

    print("✅ patterns basic checks passed!")
except AssertionError:
    print("❌ patterns failed a check. Review your implementation.")
except NotImplementedError as e:
    print(e)


✅ patterns basic checks passed!


---

## Problem 3 — `seed_board(name: str, size=32)`

### Goal & Spec
Create a fresh board with `make_board(size)` and place or generate the requested pattern by name.

- If the name is not `"random"`, Look up `name` in `patterns()`.
- If the value is a **NumPy array** (fixed pattern), place it **centered** on the new board using `center_place(board, pattern)` (implemented in the next problem).

> **Note:** This function depends on `center_place`, which you'll implement **next**. The checks here will run **after** you complete Problem 4.


In [ ]:
def seed_board(name: str, size: int = 32) -> np.ndarray:
    """Return a new (size x size) board seeded with the named pattern.

    - Fixed pattern (ndarray): center it on a blank board.
    - Generated pattern (callable): call with shape=(size, size) and return the result.
    """
    pats = patterns()
    pattern = pats[name]

    if callable(pattern):
        # Generated pattern (e.g., "random")
        return pattern((size, size))
    else:
        # Fixed pattern — center it on a new board
        board = np.zeros((size, size), dtype=int)
        p_rows, p_cols = pattern.shape

        # Compute top-left corner for centering
        start_row = (size - p_rows) // 2
        start_col = (size - p_cols) // 2

        board[start_row:start_row + p_rows, start_col:start_col + p_cols] = pattern
        return board


---

## Problem 4 — `center_place(board, pattern)`

### Goal & Spec
Copy the smaller `pattern` array into the **center** of the larger `board` array.

- Let `R, C = board.shape`; `r, c = pattern.shape`.

- Compute the **start row/col** as integer midpoints:
  - Example if a board is 16 cells wide and a pattern is 8 cells wide, the pattern should begin in the 5th row.  This would leave 4 rows above, and 4 rows below the placed pattern.

- Copy the pattern into the board using slicing:

- Return the modified `board` (also modified in place).

### Starter Code


In [ ]:
def center_place(board: np.ndarray, pattern: np.ndarray) -> np.ndarray:
    """Place pattern into the center of board and return the board."""
    R, C = board.shape
    r, c = pattern.shape

    start_row = (R - r) // 2
    start_col = (C - c) // 2

    board[start_row:start_row + r, start_col:start_col + c] = pattern

    return board
    raise NotImplementedError("center_place is not yet implemented.")


### Quick Checks (Problem 3 & 4)
Now that `center_place` exists, we can check both `center_place` and `seed_board`.

- For a simple **block** pattern centered on a 6x6 board.
- For a **random** full-board seed of shape (8,8).


In [ ]:
# Create a simple block pattern for testing
_block = np.array([[1,1],[1,1]], dtype=int)

try:
    b = make_board(6)
    b = center_place(b, _block)
    # Expect the 2x2 block at rows 2..3 and cols 2..3 (0-indexed)
    assert b.sum() == 4
    assert b[2:4, 2:4].sum() == 4

    # seed_board with a fixed pattern (using provided 'loaf' from patterns)
    sb = seed_board('loaf', size=10)
    assert sb.shape == (10,10)
    assert sb.sum() == np.array([
        [0,1,1,0],
        [1,0,0,1],
        [0,1,0,1],
        [0,0,1,0]
    ], dtype=int).sum()

    # seed_board with random (callable)
    sb_r = seed_board('random', size=8)
    assert sb_r.shape == (8,8)
    assert set(np.unique(sb_r)).issubset({0,1})

    print("✅ center_place & seed_board basic checks passed!")
except AssertionError:
    print("❌ A check failed. Review your implementations for Problems 3 & 4.")
except NotImplementedError as e:
    print(e)
except ValueError as e:
    print("ValueError:", e)


✅ center_place & seed_board basic checks passed!


---

## Problem 5 — `iterate(population)`

### Goal & Spec
Advance the board `population` by **one** Game of Life step, **in place**, and return it. Use a *fixed zero border* (do **not** write to the outermost rows/cols).

**Rules** for each *inner* cell (1 = alive, 0 = dead), based on the sum of its 8 neighbors `N`:
- **Birth:** dead cell with `N == 3` becomes 1
- **Survival:** alive cell with `N == 2 or `N == 3` stays 1
- **Otherwise:** becomes 0

### How to think about neighbor slicing (intuition)
Imagine the *inner* window `population[1:-1, 1:-1]` as the cells you're going to update. Each of those cells has eight neighbors: top-left, top, top-right, left, right, bottom-left, bottom, bottom-right. You can obtain all eight neighbor fields **at once** by slicing shifted views of the array and making sure they all have the **same shape** as the inner window. For example, the "top" neighbors for every inner cell are `population[0:-2, 1:-1]`, which is the array one row above the inner window (same columns). Likewise the bottom neighbors are `population[2:, 1:-1]`, and so on. When you sum these eight arrays elementwise, you get an array `N` of the same shape as the inner window where each entry counts live neighbors for that inner cell.

### Masks and comparisons in NumPy
A **mask** is a boolean array created by comparing arrays elementwise (e.g., `N == 3`). You can combine masks with bitwise operators `&` (and) and `|` (or). For example:
- `isDead = (population[1:-1, 1:-1] == 0)`
Then build rule masks:
- `birth  = has3N & isDead`


Finally, **update only the inner window** using these masks. Because all masks have the same shape as `population[1:-1, 1:-1]`, you can zero that inner window and set ones where birth or survive is `True`.

### Vectorized Plan (no Python loops):
1. Slice the eight neighbor fields (top-left, top, top-right, left, right, bottom-left, bottom, bottom-right) using `Z[<rows>, <cols>]` slices.
2. Sum those eight arrays into `N`.
3. Build boolean masks for `birth` and `survive` using comparisons and bitwise operators (`&`, `|`).
4. Update the *inner window* in place: zero it, then set `1` only where birth or survive is `True` (leave the border as zeros).
5. return the updated population


In [ ]:
def iterate(population: np.ndarray) -> np.ndarray:
    """Compute one in-place Life step on population with a fixed zero border, then return it."""
    # 1
    N = (
        population[:-2, :-2] + population[:-2, 1:-1] + population[:-2, 2:] +
        population[1:-1, :-2]                      + population[1:-1, 2:] +
        population[2:, :-2] + population[2:, 1:-1] + population[2:, 2:]
    )

    # 2
    inner = population[1:-1, 1:-1]

    # 3
    birth = (inner == 0) & (N == 3)                # Dead cell with exactly 3 neighbors → live
    survive = (inner == 1) & ((N == 2) | (N == 3)) # Live cell with 2 or 3 neighbors → stays live

    # 4
    inner[...] = 0
    inner[birth | survive] = 1

    #5
    return population


### Quick Checks
- A centered 2x2 **block** should remain unchanged after one step.
- A horizontal **blinker** (1x3) should flip to vertical (3x1) after one step.


In [ ]:
try:
    # Stable 2x2 block
    Z = make_board(6)
    Z[2:4, 2:4] = 1
    iterate(Z)
    assert Z[2:4, 2:4].sum() == 4 and Z.sum() == 4

    # Blinker oscillation
    Z = make_board(5)
    # Horizontal at row 2, cols 1..3 (0-indexed)
    Z[2, 1:4] = 1
    iterate(Z)
    # Expect vertical at col 2, rows 1..3
    assert Z[1:4, 2].sum() == 3 and Z.sum() == 3

    print("✅ iterate basic checks passed!");
except AssertionError:
    print("❌ iterate failed a check. Review your implementation.");
except NotImplementedError as e:
    print(e)


✅ iterate basic checks passed!


---

## Render Function (Do Not Modify)

This function is responsible for rendering the grid.  

In [ ]:
def game_of_life(pattern="pulsar", fps=8, steps=200, size=32, alive="██", dead="  "):
    population = seed_board(pattern, size=size)
    # Sized container avoids any layout shift between frames
    w = HTML(layout=Layout(margin='0', padding='0'))
    display(w)

    sleep_duration = 1.0 / max(1, int(fps))
    i = 0
    try:
        while steps is not None and i < steps:
            # build HTML once per frame
            txt = "\n".join("".join(alive if c else dead for c in row) for row in population)
            w.value = f"<pre style='line-height:1; font-family:monospace; font-size:12px; margin:0'>{txt}</pre>"
            iterate(population)
            i += 1
            time.sleep(sleep_duration)
    except KeyboardInterrupt:
        pass


---

## Demo

Once you've completed all problems, try the animation:

```python
game_of_life(pattern="pulsar", fps=8, steps=120, size=32)
game_of_life(pattern="random", fps=6, steps=120, size=32)
```
